# Dataset Understanding

This section inspects the CreditSense dataset and data dictionary before any cleaning or preprocessing is performed. The aim is to understand the structure of the data, identify feature types, and note any fields that require special treatment later.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/creditsense_applications.csv")
dictionary = pd.read_csv("../data/creditsense_data_dictionary.csv")

print(df.shape)
df.head()

(6000, 27)


,application_id,age_years,sex,marital_status,residency_status,num_dependents,region,employment_status,months_in_current_job,annual_income_gbp,...,has_savings_account,savings_balance_gbp,checking_account_status,loan_amount_gbp,loan_term_months,loan_purpose,has_guarantor,previous_application_declined,application_channel,default_within_24m
0,APP-000001,34,Female,Married,Settled Status,1,North West,Permanent,19,66550,...,Yes,380.0,500 GBP or more,13100,36,Home improvement,No,No,Online,0
1,APP-000002,43,Female,Married,Settled Status,1,South East,Permanent,29,20820,...,No,0.0,Overdrawn,9700,60,Medical,No,No,Branch,1
2,APP-000003,47,Male,Single,UK Citizen,0,South East,Permanent,115,32710,...,No,0.0,500 GBP or more,8600,36,Home improvement,No,No,Online,0
3,APP-000004,63,Male,Married,UK Citizen,1,London,Permanent,146,47300,...,Yes,NaN,0 to 500 GBP,25600,36,Home improvement,No,No,Online,0
4,APP-000005,40,Male,Single,UK Citizen,1,Yorkshire,Permanent,22,41400,...,Yes,860.0,0 to 500 GBP,3500,36,Car,Yes,No,Branch,0


In [2]:
dictionary.head(30)

,column,type,description,protected_attribute,notes
0,application_id,string,Unique application reference,No,Identifier only - must NOT be used as a feature
1,age_years,integer,Applicant age at application,Yes,Protected characteristic under Equality Act 2010
2,sex,categorical,Female / Male,Yes,Protected characteristic under Equality Act 2010
3,marital_status,categorical,Single / Married / Divorced / Widowed,Yes,Correlates strongly with age
4,residency_status,categorical,UK Citizen / Settled Status / Visa Holder,Yes,Proxy for nationality - protected characteristic
5,num_dependents,integer,Number of financial dependents,No,Correlates with marital_status
6,region,categorical,UK region of residence,No,Can act as a proxy for other attributes
7,employment_status,categorical,Permanent / Fixed-term / Self-employed / Zero-...,No,Retired appears only in older applicants
8,months_in_current_job,integer,Tenure in current employment (months),No,Zero for unemployed applicants - structural ze...
9,annual_income_gbp,integer,Gross annual income in GBP,No,Not adjusted for region


## Initial Data Dictionary Observations

The data dictionary highlights several fields that will require careful handling during preprocessing.

- `application_id` is an identifier and will not be used as a predictive feature.
- Protected attributes are `age_years`, `sex`, `marital_status`, `residency_status` and `housing_status`.
- `region` may act as a proxy for other protected characteristics.
- `credit_history` is ordinal and will require an appropriate encoding strategy.
- `checking_account_status` is also ordinal, but the `No account` category does not fit naturally on the ordered scale.
- `months_in_current_job` contains structural zeros for unemployed applicants. These zeros represent real information rather than missing data.
- Missing values occur in `years_at_current_address` and `savings_balance_gbp`.
- `default_within_24m` is the binary target variable.

In [3]:
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = df.select_dtypes(include="str").columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['age_years', 'num_dependents', 'months_in_current_job', 'annual_income_gbp', 'monthly_expenses_gbp', 'years_at_current_address', 'existing_debt_gbp', 'num_existing_credit_lines', 'num_late_payments_24m', 'savings_balance_gbp', 'loan_amount_gbp', 'loan_term_months', 'default_within_24m']

Categorical columns:
['application_id', 'sex', 'marital_status', 'residency_status', 'region', 'employment_status', 'housing_status', 'credit_history', 'has_savings_account', 'checking_account_status', 'loan_purpose', 'has_guarantor', 'previous_application_declined', 'application_channel']


In [4]:
target_column = "default_within_24m"
id_column = "application_id"

numerical_features = [
    col for col in numerical_columns
    if col != target_column
]

categorical_features = [
    col for col in categorical_columns
    if col != id_column
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age_years', 'num_dependents', 'months_in_current_job', 'annual_income_gbp', 'monthly_expenses_gbp', 'years_at_current_address', 'existing_debt_gbp', 'num_existing_credit_lines', 'num_late_payments_24m', 'savings_balance_gbp', 'loan_amount_gbp', 'loan_term_months']

Categorical features:
['sex', 'marital_status', 'residency_status', 'region', 'employment_status', 'housing_status', 'credit_history', 'has_savings_account', 'checking_account_status', 'loan_purpose', 'has_guarantor', 'previous_application_declined', 'application_channel']


## Feature Groups

The dataset contains 25 predictive features in total.

- 12 numerical features
- 13 categorical features
- `application_id` is excluded because it is an identifier
- `default_within_24m` is the target variable

Some categorical features, particularly `credit_history` and `checking_account_status`, will require more careful encoding because they have an ordinal structure.

In [5]:
print(df["credit_history"].value_counts())

print("\nChecking account status:")
print(df["checking_account_status"].value_counts())

credit_history
Existing credits paid    1935
All paid on time         1727
Past delays              1356
No credits taken          882
Critical account          100
Name: count, dtype: int64

Checking account status:
checking_account_status
0 to 500 GBP       2185
500 GBP or more    1971
Overdrawn          1252
No account          592
Name: count, dtype: int64


## Ordinal Features

Two categorical features require special consideration because they have an ordinal structure.

### `credit_history`
The data dictionary identifies `credit_history` as ordinal. Its categories represent different types of previous credit behaviour. The exact encoding order will be decided during preprocessing rather than treating it as a normal unordered category.

### `checking_account_status`
This field contains an ordered financial scale:

- `Overdrawn`
- `0 to 500 GBP`
- `500 GBP or more`

However, `No account` does not fit naturally onto that scale. This will require a separate preprocessing decision rather than applying a simple ordinal encoding to all four categories.

In [6]:
protected_attributes = dictionary[
    dictionary["protected_attribute"] == "Yes"
][["column", "description", "notes"]]

protected_attributes

,column,description,notes
1,age_years,Applicant age at application,Protected characteristic under Equality Act 2010
2,sex,Female / Male,Protected characteristic under Equality Act 2010
3,marital_status,Single / Married / Divorced / Widowed,Correlates strongly with age
4,residency_status,UK Citizen / Settled Status / Visa Holder,Proxy for nationality - protected characteristic
11,housing_status,Owner / Mortgage / Renting / Living with family,Treated as protected for this assessment


## Protected Attributes and Proxy Risk

The data dictionary marks the following fields as protected attributes for this assessment:

- `age_years`
- `sex`
- `marital_status`
- `residency_status`
- `housing_status`

The `region` field is not marked as protected, but the data dictionary notes that it may act as a proxy for other protected characteristics.

These fields will require careful consideration during model design and fairness evaluation. No decision has been made yet about whether protected attributes will be included in the final predictive model.